# CTA — Gate 2: decode / KV-cache validation

**Kill-test:** everything proven so far (1.83× prefill on Qwen2.5-3B, collapse-ratio on RAG/code) is **prefill-only**. Products live on **decode** — autoregressive generation over a KV-cache.

The question this notebook answers: after we collapse a prompt (`n → m`) and build a KV-cache over the **m** pooled positions, can we generate **coherent** continuations on top of that compressed cache, and is decode actually **faster / lighter**?

Three outcomes (fixed in advance):
- **GO** — coherent output, quality ≈ baseline, decode faster (attention over m<n). Product lives on the whole serving loop.
- **PARTIAL** — quality degrades but prefill win is real → honest `prefill-accelerator` positioning.
- **NO-GO** — generation over collapsed cache breaks (incoherent / must un-collapse → win erased).

Run top-to-bottom on a **T4** (free Colab). ~2-3 min.

## 0 · Environment check

In [ ]:
import torch, sys
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

## 1 · Install deps + fetch CTA source

In [ ]:
import os, subprocess
def sh(cmd):
    print('$', cmd); return subprocess.run(cmd, shell=True).returncode
sh('pip -q install -U transformers accelerate >/dev/null 2>&1')
if not os.path.isdir('compositional-token-algebra'):
    sh('git clone -q https://github.com/VadShv/compositional-token-algebra')
sys.path.insert(0, '/content/compositional-token-algebra/src')
print('ok')

## 2 · Load model (Qwen2.5-3B, bf16, FlashAttention/SDPA)
Same backbone as the prefill test. Auto-falls back to 0.5B if VRAM is tight.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = 'Qwen/Qwen2.5-3B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,
              attn_implementation='sdpa').to(DEVICE)
except Exception as e:
    print('3B failed, falling back to 0.5B:', str(e)[:100])
    MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,
              attn_implementation='sdpa').to(DEVICE)
model.eval()
cfg = model.config
print(f'model={MODEL_NAME} layers={cfg.num_hidden_layers} d={cfg.hidden_size} device={DEVICE} dtype={DTYPE}')

## 3 · Helpers: collapse prompt, generate over KV-cache, judge quality
- `collapse_prompt_embeds` — exact detector → compose spans → pooled embeds `[m, d]`.
- `generate_from_embeds` — prefill with `use_cache=True` → KV-cache of length `L` → greedy decode `N_GEN` tokens, `position_ids` continuing from `L` (honest compressed-position setup, same anchor convention as prefill).
- `ppl_of_continuation` — the **honest judge**: perplexity of the generated text under the FULL (uncollapsed) model given the FULL prompt.

In [ ]:
import time, json, math, random
from cta.detector import select_collapse_spans, build_segments
from cta.algebra import compose
N_GEN = 64

def emb_of(ids):
    return model.model.embed_tokens(ids)

@torch.no_grad()
def collapse_prompt_embeds(ids, mode='norm'):
    emb = emb_of(ids)
    spans = select_collapse_spans(ids.tolist(), k_min=3, k_max=16, f_min=2)
    segs = build_segments(ids.tolist(), spans)
    vecs = []
    for (s, e, is_col) in segs:
        vecs.append(compose(emb[s:e], mode=mode)[0] if is_col else emb[s])
    return torch.stack(vecs, 0), len(ids), len(segs)

@torch.no_grad()
def generate_from_embeds(inputs_embeds, n_gen):
    L = inputs_embeds.shape[0]
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time()
    out = model.model(inputs_embeds=inputs_embeds.unsqueeze(0), use_cache=True)
    past = out.past_key_values
    nid = int(model.lm_head(out.last_hidden_state[:, -1, :]).argmax(-1))
    if DEVICE=='cuda': torch.cuda.synchronize()
    prefill = (time.time()-t0)*1000
    gen=[nid]; pos=L
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0=time.time()
    for _ in range(n_gen-1):
        te = model.model.embed_tokens(torch.tensor([[nid]], device=DEVICE)).to(inputs_embeds.dtype)
        pid = torch.tensor([[pos]], device=DEVICE)
        out = model.model(inputs_embeds=te, past_key_values=past, use_cache=True, position_ids=pid)
        past = out.past_key_values
        nid = int(model.lm_head(out.last_hidden_state[:, -1, :]).argmax(-1))
        gen.append(nid); pos+=1
    if DEVICE=='cuda': torch.cuda.synchronize()
    per_tok = (time.time()-t0)/max(1,(n_gen-1))*1000
    return gen, prefill, per_tok, past

@torch.no_grad()
def ppl_of_continuation(prompt_ids, gen_ids):
    full = torch.cat([prompt_ids, torch.tensor(gen_ids, device=DEVICE)])
    out = model.model(inputs_embeds=emb_of(full).unsqueeze(0))
    logits = model.lm_head(out.last_hidden_state)[0]
    n=len(prompt_ids); tgt=torch.tensor(gen_ids, device=DEVICE)
    lp = torch.log_softmax(logits[n-1:n-1+len(gen_ids)].float(), -1)
    ll = lp[torch.arange(len(gen_ids)), tgt]
    return float(torch.exp(-ll.mean()))

def kv_positions(past):
    try: return past.get_seq_length()
    except Exception:
        try: return past[0][0].shape[-2]
        except Exception: return None
print('helpers ready')

## 4 · RAG-like prompt (real internal repetition) + run A vs B
A = baseline full-prompt decode. B = CTA collapsed-prompt decode.

In [ ]:
def make_rag_prompt():
    doc=('Retrieved document. User {u} logged in from 10.0.{a}.{b} at 2026-07-{d}. '
         'Session established. Role=admin, region=eu-west. Auth method oauth2. '
         'Token TTL 3600s. Rate limit 1000 req/min. Endpoint /api/v2/login returned 200 OK. ')
    random.seed(0); ctx=''
    for i in range(10):
        ctx += doc.format(u=1000+i, a=random.randint(0,255), b=random.randint(0,255), d=random.randint(10,28))
    return ctx + ('\n\nBased on the retrieved documents above, summarize the common '
                  'authentication configuration in one sentence:')

prompt = make_rag_prompt()
ids = tok(prompt, add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
n = len(ids)
a_ids, a_prefill, a_pertok, a_past = generate_from_embeds(emb_of(ids), N_GEN)
collapsed, _, m = collapse_prompt_embeds(ids)
b_ids, b_prefill, b_pertok, b_past = generate_from_embeds(collapsed, N_GEN)
print(f'n={n}  m={m}  collapse_ratio={m/n:.3f}')

## 5 · Metrics: coherence, quality, speed, memory

In [ ]:
match = sum(1 for x,y in zip(a_ids,b_ids) if x==y)/len(a_ids)
a_text = tok.decode(a_ids); b_text = tok.decode(b_ids)
a_ppl = ppl_of_continuation(ids, a_ids)
b_ppl = ppl_of_continuation(ids, b_ids)
res = dict(model=MODEL_NAME, n_prompt=n, m_collapsed=m, collapse_ratio=round(m/n,4), n_gen=N_GEN,
    baseline=dict(prefill_ms=round(a_prefill,1), per_tok_ms=round(a_pertok,2),
                  kv_positions=kv_positions(a_past), ppl_self=round(a_ppl,3)),
    cta=dict(prefill_ms=round(b_prefill,1), per_tok_ms=round(b_pertok,2),
             kv_positions=kv_positions(b_past), ppl_under_full_model=round(b_ppl,3)),
    greedy_match_rate=round(match,3), baseline_text=a_text, cta_text=b_text)
print(json.dumps({k:v for k,v in res.items() if k not in ('baseline_text','cta_text')}, indent=2))
print('\n--- BASELINE continuation ---\n', a_text)
print('\n--- CTA-DECODE continuation ---\n', b_text)

## 6 · Save results → send `results_gate2.json` back

In [ ]:
with open('results_gate2.json','w') as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print('saved results_gate2.json')
try:
    from google.colab import files; files.download('results_gate2.json')
except Exception: pass

## How to read the result (go / no-go for Gate 2)

- **GO** — CTA-decode text is coherent AND `ppl_under_full_model` is close to baseline `ppl_self` (say within ~1.5×) AND `cta.per_tok_ms` ≤ baseline. Decode survives the compressed cache.
- **PARTIAL** — text readable but PPL noticeably worse, or decode not faster → position CTA as a **prefill accelerator** only.
- **NO-GO** — CTA-decode text is incoherent / PPL blows up (like the keep-pooled +6810% regime) → generation over collapsed KV is broken; product limited to offline/prefill metrics.

`kv_positions` shows the memory story: baseline caches `n` positions, CTA caches `m` — the KV-cache is physically smaller, which is the decode-time memory win **if** quality holds.